# E3 — Bid-Ask Analysis W4 (ESM5/ESU5)

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [ ]:
wm = WINDOWS_META['W4']
summary_rows = []
for day in wm['days']:
    for sym in [wm['front'], wm['back']]:
        fpath = SEAGATE_DIR / f'mbp10_{sym}_{wm["front"]}_{wm["back"]}_{day}.parquet'
        if not fpath.exists():
            # try alternate naming convention
            fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
            fpath = fpath[0] if fpath else None
        if fpath is None:
            print(f'  MISSING: {sym} {day}'); continue
        df = pd.read_parquet(fpath, columns=['ts_event','symbol','bid_px_00','ask_px_00'])
        df['ts_event'] = pd.to_datetime(df['ts_event'])
        # RTH filter
        rth_mask = ((df['ts_event'].dt.hour * 60 + df['ts_event'].dt.minute) >= wm['rth_start_min']) & \
                   ((df['ts_event'].dt.hour * 60 + df['ts_event'].dt.minute) <= wm['rth_end_min'])
        df = df[rth_mask]
        df['ba'] = (df['ask_px_00'] - df['bid_px_00']) / TICK
        summary_rows.append(dict(
            day=day, symbol=sym,
            mean=df['ba'].mean(), median=df['ba'].median(),
            pct_1t=(df['ba'] == 1).mean(), pct_2t=(df['ba'] == 2).mean(),
            pct_3t=(df['ba'] >= 3).mean(),
        ))
        del df; gc.collect()

ba_df = pd.DataFrame(summary_rows)
print(ba_df.to_string(index=False))


In [ ]:
# Bar chart: mean BA per day (front vs back)
wm = WINDOWS_META['W4']
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(wm['days']))
w = 0.35
front_means = ba_df[ba_df['symbol']==wm['front']].set_index('day')['mean'].reindex(wm['days'])
back_means  = ba_df[ba_df['symbol']==wm['back']].set_index('day')['mean'].reindex(wm['days'])
ax.bar(x-w/2, front_means.values, w, label=wm['front'], color='steelblue', alpha=0.85)
ax.bar(x+w/2, back_means.values,  w, label=wm['back'],  color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(wm['day_labels'])
ax.set_ylabel('Mean BA Spread (ticks)')
ax.set_title(f'W4: Mean Bid-Ask Spread per Day')
ax.legend()
save_fig(fig, 'E4_ba_spread_w4.png')


In [ ]:
# Stacked bar: tick distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, sym in zip(axes, [wm['front'], wm['back']]):
    sub = ba_df[ba_df['symbol']==sym].set_index('day').reindex(wm['days'])
    x = np.arange(len(wm['days']))
    ax.bar(x, sub['pct_1t'].values, label='1-tick', color='#2ecc71', alpha=0.85)
    ax.bar(x, sub['pct_2t'].values, bottom=sub['pct_1t'].values, label='2-tick', color='#3498db', alpha=0.85)
    ax.bar(x, sub['pct_3t'].values, bottom=(sub['pct_1t']+sub['pct_2t']).values, label='3+tick', color='#e74c3c', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(wm['day_labels'])
    ax.set_title(f'{sym} BA width distribution')
    ax.set_ylabel('Fraction of quotes')
    ax.legend(fontsize=8)
fig.suptitle(f'W4 Bid-Ask Width Distribution', fontsize=12)
save_fig(fig, 'E4_ba_width_dist_w4.png')
